# Template application

In [ ]:
template_file = 'entity.templ.html'

In [ ]:
import re
import logging
log = logging.getLogger(__name__)

In [ ]:
import xml.dom.minidom
import IPython

def beautify_xml(flat_xml):
    try:
        wrapped = '<root doc="container to properly encapsulate xml">\n{}\n</root>'.format(flat_xml)
        dom = xml.dom.minidom.parseString(wrapped)
        return dom.toprettyxml()
    except xml.parsers.expat.ExpatError as e:
        from lxml import etree
        parser = etree.XMLParser(dtd_validation=True)
        etree.fromString(wrapped, parser)
        m = re.search('line ([0-9]+), column ([0-9]+)', str(e))
        message = 'Malformed xml ' + flat_xml[:50] + ' ...'
        if m:
            lines = wrapped.splitlines()
            messaage = 'Malformed xml input on position {} in: {}'.format(m.group(2), lines[int(m.group(1))-1])
        else:
            message = message + flat_xml[:50] + ' ...'
        raise Exception(message)
    
beautify_xml('<a></a>')
try:
    beautify_xml('<b></a>')
    assert false, 'Should fail'
except Exception as e:
    print('works')

# Load datasource

In [ ]:
import json

content = None
with open('testdata/IM-sample.json', 'r') as source:
     content = json.load(source)
        
print('Model' + content['model']['name'])
entities = list(content['entities'])
first_entity_name = 'ENTI12701'

entities[:3]

In [ ]:
from xml.sax.saxutils import escape

class Toolbox:
    
    def __init__(self, language='de'):
        self.language = language
    
    def translate(self, field):
        return escape(field[self.language])
    
    def href(self, element):
        '''Returns the url of an element'''
        return str(element)

In [ ]:
first = content['entities'][first_entity_name]
log.warning('Working with entity ' + first_entity_name + ": " + Toolbox('de').translate(first['name']))
first

In [ ]:
entity_template = None
with open('./templates/' + template_file, 'r') as f:
    entity_template = f.read()

assert entity_template

IPython.display.Code(entity_template)

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)

template = env.get_template(template_file)
rendered_template = template.render(entity=first, util=Toolbox('de'))
html = re.sub('<!--.+?->(\n+)*', '', rendered_template) # strip comment lines
IPython.display.Code(html)

In [ ]:
IPython.display.HTML(html)

In [ ]:
IPython.display.Code(beautify_xml(html))